# 09. Spatial validation and leakage

Rivers are not independent rows. Water bodies on the same river share upstream pressures, geology,
climate and monitoring teams. A random train/test split puts neighbours on both sides, and a flexible model
can score well by recognising *places* rather than learning *relationships*.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3})
from river_health import modeling as M
t = M.prepare(pd.read_parquet(ROOT / "data" / "processed" / "river_features.parquet"))
# a small, readable working set used throughout
F = ["up_stw_dwf_m3d_per_km2", "up_spill_hours_per_km2", "up_lc_urban", "up_lc_arable",
     "up_lc_improved_grass", "up_lc_seminatural", "up_area_km2", "local_n_private_sewage"]
d = t[t.phos_good.notna()].reset_index(drop=True)
X, y = d[F].to_numpy(float), d.phos_good.to_numpy(int)
groups = d.management_catchment.to_numpy()
print(X.shape, "share with good phosphate:", y.mean().round(3))

(3706, 8) share with good phosphate: 0.484


## How similar are neighbours?

In [2]:
from scipy.spatial import cKDTree
xy = d[["easting", "northing"]].to_numpy(); tree = cKDTree(xy)
dist, idx = tree.query(xy, k=6)
same = (y[idx[:, 1:]] == y[:, None]).mean()
print(f"share of 5 nearest neighbours with the same phosphate outcome: {same:.2f} (chance: {y.mean()**2 + (1-y.mean())**2:.2f})")

share of 5 nearest neighbours with the same phosphate outcome: 0.72 (chance: 0.50)


## Random vs spatial folds

In [3]:
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.model_selection import KFold, StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
Xl = np.c_[X, d[["easting", "northing"]].to_numpy()]      # coordinates make the leak visible
schemes = {"random 5-fold": list(KFold(5, shuffle=True, random_state=0).split(Xl)),
           "spatial 5-fold": list(StratifiedGroupKFold(5, shuffle=True, random_state=0).split(Xl, y, groups))}
models = {"logistic": lambda: make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), LogisticRegression(max_iter=2000)),
          "LightGBM": lambda: lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63, min_child_samples=5, verbose=-1)}
rows = []
for feats, XX in [("pressures", X), ("pressures + coordinates", Xl)]:
    for sname, folds in schemes.items():
        for mname, mk in models.items():
            if mname == "logistic" and feats != "pressures": continue
            o = np.zeros(len(y))
            for a, b in folds: o[b] = mk().fit(XX[a], y[a]).predict_proba(XX[b])[:, 1]
            rows.append((feats, mname, sname, roc_auc_score(y, o)))
r = pd.DataFrame(rows, columns=["features", "model", "cv", "AUC"])
r.pivot_table(index=["features", "model"], columns="cv", values="AUC").round(3)

cv                                random 5-fold  spatial 5-fold
features                model                                  
pressures               LightGBM          0.816           0.801
                        logistic          0.793           0.790
pressures + coordinates LightGBM          0.878           0.839

Neighbouring rivers share their phosphate outcome 72% of the time, against 50% by chance.
With random folds, LightGBM given coordinates scores 0.878. With whole catchments held out it scores 0.839:
about half of the coordinates' apparent gain was the model recognising places. The other half is regional
structure the pressures miss (soils, rainfall, livestock). Note also that random folds flatter LightGBM
(+0.015) much more than the logistic model (+0.003). The question the project answers is *"what happens in a river the model
has not seen?"*, so all reported scores use spatial folds.

## Target leakage: features that are the target in disguise

The phosphate **class** is defined by comparing measured phosphate with a standard for the river type. Using
measured phosphate as a feature is close to using the label.

In [4]:
chem = d.chem_orthophosphate_mgP.to_numpy()
folds = schemes["spatial 5-fold"]
for lab, XX in [("pressures only", X), ("+ measured phosphate", np.c_[X, chem])]:
    o = np.zeros(len(y))
    for a, b in folds: o[b] = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=15, verbose=-1).fit(XX[a], y[a]).predict_proba(XX[b])[:, 1]
    print(f"{lab:22s} AUC {roc_auc_score(y, o):.3f}")

pressures only         AUC 0.808


+ measured phosphate   AUC 0.894


AUC jumps from 0.81 to 0.89. A big jump is not a better model of *why* rivers fail. It answers a different, nearly circular question.
The project therefore models from **pressures**, which are things a regulator can act on, and uses chemistry only
as a diagnostic.

## Other leaks to watch for in this kind of data
* **Time.** Pressures must precede the outcome. Spill features use 2021-24 for the 2025 classification;
  2025 spills are kept aside.
* **Grouping variables as features.** Management catchment defines the CV groups; target-encoding it would
  smuggle in neighbours' labels.
* **Preprocessing on all data.** Imputation, scaling and encoding are fitted inside each training fold
  (scikit-learn pipelines).